In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# **Verification Step**

In [5]:
dataset1 = pd.read_csv("../Data/Raw/deceptive-opinion.csv", engine='python', on_bad_lines='skip')
dataset2 = pd.read_csv("../Data/Raw/Fake reviews dataset.csv")
dataset3 = pd.read_csv("../Data/Raw/new_data_train.csv", sep='\t')
dataset4 = pd.read_csv("../Data/Raw/new_data_test.csv", sep='\t')

In [7]:
dataset1.head(1)

,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...


In [8]:
# 1. Check shapes
print(f"Dataset 1 Shape: {dataset1.shape}")
print(f"Dataset 2 Shape: {dataset2.shape}")
print(f"Dataset 3 Shape: {dataset3.shape}")
print(f"Dataset 4 Shape: {dataset4.shape}")

# 2. Check Dataset 1 - Is there any label column?
print("\n--- Dataset 1 Head ---")
display(dataset1[['deceptive', 'text']].head(2))

# 3. Check unique values of labels in Dataset 2, 3, 4
print("\n--- Dataset 2 Labels ---")
print(dataset2['label'].value_counts(dropna=False))

print("\n--- Dataset 3 Labels ---")
print(dataset3['flagged'].value_counts(dropna=False))

print("\n--- Dataset 4 Labels ---")
print(dataset4['flagged'].value_counts(dropna=False))

Dataset 1 Shape: (1600, 5)
Dataset 2 Shape: (40432, 4)
Dataset 3 Shape: (9929, 23)
Dataset 4 Shape: (2483, 23)

--- Dataset 1 Head ---


,deceptive,text
0,truthful,We stayed for a one night getaway with family ...
1,truthful,Triple A rate with upgrade to view room was le...



--- Dataset 2 Labels ---
label
CG    20216
OR    20216
Name: count, dtype: int64

--- Dataset 3 Labels ---
flagged
0    4994
1    4935
Name: count, dtype: int64

--- Dataset 4 Labels ---
flagged
1    1271
0    1212
Name: count, dtype: int64


# **Processing & Merging**

In [9]:
import pandas as pd

# 1. Process Dataset 2 (Kaggle Fake Reviews)
# Keep only text and label, rename to standard
df2 = dataset2[['text_', 'label']].rename(columns={'text_': 'review'})
# Map 'CG' (Computer Generated) to 1 and 'OR' (Original) to 0
df2['label'] = df2['label'].map({'CG': 1, 'OR': 0})

# 2. Process Dataset 3 (Train Yelp data)
# Keep only reviewContent and flagged, rename to standard
df3 = dataset3[['reviewContent', 'flagged']].rename(columns={'reviewContent': 'review', 'flagged': 'label'})
# Ensure labels are numeric (assuming 'Y'/'N' or already 1/0)
# If they are 'Y'/'N', uncomment the next line:
# df3['label'] = df3['label'].map({'Y': 1, 'N': 0})

# 3. Process Dataset 4 (Test Yelp data)
# Keep only reviewContent and flagged, rename to standard
df4 = dataset4[['reviewContent', 'flagged']].rename(columns={'reviewContent': 'review', 'flagged': 'label'})
# If they are 'Y'/'N', uncomment the next line:
# df4['label'] = df4['label'].map({'Y': 1, 'N': 0})

df1 = dataset1[['text', 'deceptive']].rename(columns={'text': 'review', 'deceptive': 'label'})

# Map 'deceptive' (Fake) to 1 and 'truthful' (Genuine) to 0
df1['label'] = df1['label'].map({'deceptive': 1, 'truthful': 0})

# Check the result
display(df1.head())
print(df1['label'].value_counts(dropna=False))
# 4. Merge Vertically
merged_df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Verify the result
print(f"Merged Shape: {merged_df.shape}")
print(f"Missing values:\n{merged_df.isna().sum()}")
display(merged_df.head())

,review,label
0,We stayed for a one night getaway with family ...,0
1,Triple A rate with upgrade to view room was le...,0
2,This comes a little late as I'm finally catchi...,0
3,The Omni Chicago really delivers on all fronts...,0
4,I asked for a high floor away from the elevato...,0


label
0    800
1    800
Name: count, dtype: int64
Merged Shape: (54444, 2)
Missing values:
review    3
label     0
dtype: int64


,review,label
0,We stayed for a one night getaway with family ...,0
1,Triple A rate with upgrade to view room was le...,0
2,This comes a little late as I'm finally catchi...,0
3,The Omni Chicago really delivers on all fronts...,0
4,I asked for a high floor away from the elevato...,0


# **Final Class Distribution**

In [10]:
# 2. Check initial shape
print(f"Initial row count: {merged_df.shape[0]}")

# 3. Handle Missing Values
# Drop rows where the review text or label is missing
merged_df = merged_df.dropna(subset=['review', 'label'])
print(f"Row count after dropping nulls: {merged_df.shape[0]}")

# 4. Handle Duplicates
# Count how many exact duplicate reviews exist
duplicate_count = merged_df.duplicated(subset=['review']).sum()
print(f"Exact duplicate reviews found: {duplicate_count}")

# Drop the duplicates (keep the first occurrence)
merged_df = merged_df.drop_duplicates(subset=['review'], keep='first')
print(f"Final clean row count: {merged_df.shape[0]}")

# 5. Check Final Class Distribution
print("\nFinal Label Distribution (1 = Fake, 0 = Genuine):")
print(merged_df['label'].value_counts(normalize=True) * 100) # Shows percentage

Initial row count: 54444
Row count after dropping nulls: 54441
Exact duplicate reviews found: 38
Final clean row count: 54403

Final Label Distribution (1 = Fake, 0 = Genuine):
label
0    50.024815
1    49.975185
Name: proportion, dtype: float64


# **Exporting it in Processed folder**

In [11]:
import os

# Create the Processed directory if it doesn't exist
output_dir = "../Data/Processed"
os.makedirs(output_dir, exist_ok=True)

# Save the cleaned dataframe
output_path = os.path.join(output_dir, "cleaned_reviews.csv")
merged_df.to_csv(output_path, index=False)

print(f"Dataset successfully saved to: {output_path}")
print(f"Total rows saved: {merged_df.shape[0]}")

Dataset successfully saved to: ../Data/Processed\cleaned_reviews.csv
Total rows saved: 54403
